# Cortex Code CLI — Usage & Cost Analysis

Queries `SNOWFLAKE.ACCOUNT_USAGE.CORTEX_CODE_CLI_USAGE_HISTORY` to surface usage patterns, credit consumption, and projected costs.

**AI Credit price:** $2.00 per credit (global on-demand, April 2026 consumption table).  
Capacity customers: adjust the `ai_credit_price` variable in cell 2 to your contracted rate.

**Required privilege:**
```sql
GRANT IMPORTED PRIVILEGES ON DATABASE SNOWFLAKE TO ROLE <your_role>;
```

In [ ]:
ai_credit_price = 2.00

print(f"AI Credit price: ${ai_credit_price:.2f} per credit")
print("Adjust ai_credit_price above if you are on Capacity with a discounted rate.")

## 1. Daily Usage Summary
Request counts, tokens, credits, and estimated dollar cost per day — last 30 days.

In [ ]:
%%sql -r daily_usage
SELECT
    USAGE_TIME::DATE                        AS usage_date,
    COUNT(*)                                AS total_requests,
    SUM(TOKENS)                             AS total_tokens,
    ROUND(SUM(TOKEN_CREDITS), 4)            AS total_credits,
    ROUND(SUM(TOKEN_CREDITS) * 2.00, 2)     AS estimated_cost_usd
FROM SNOWFLAKE.ACCOUNT_USAGE.CORTEX_CODE_CLI_USAGE_HISTORY
GROUP BY 1
ORDER BY 1 DESC
LIMIT 30

## 2. Weekly Usage Trend
Aggregates by week — useful for spotting adoption ramp and seasonal patterns.

In [ ]:
%%sql -r weekly_trend
SELECT
    DATE_TRUNC('WEEK', USAGE_TIME)          AS week_start,
    COUNT(*)                                AS total_requests,
    SUM(TOKENS)                             AS total_tokens,
    ROUND(SUM(TOKEN_CREDITS), 4)            AS total_credits,
    ROUND(SUM(TOKEN_CREDITS) * 2.00, 2)     AS estimated_cost_usd,
    ROUND(AVG(TOKENS), 0)                   AS avg_tokens_per_request
FROM SNOWFLAKE.ACCOUNT_USAGE.CORTEX_CODE_CLI_USAGE_HISTORY
GROUP BY 1
ORDER BY 1 DESC
LIMIT 13

## 3. Top Users by Credits
Ranks all users by total AI credit consumption, with first/last usage timestamps.

In [ ]:
%%sql -r top_users
SELECT
    USER_ID,
    COUNT(*)                                AS total_requests,
    SUM(TOKENS)                             AS total_tokens,
    ROUND(SUM(TOKEN_CREDITS), 4)            AS total_credits,
    ROUND(SUM(TOKEN_CREDITS) * 2.00, 2)     AS estimated_cost_usd,
    MIN(USAGE_TIME)                         AS first_usage,
    MAX(USAGE_TIME)                         AS last_usage
FROM SNOWFLAKE.ACCOUNT_USAGE.CORTEX_CODE_CLI_USAGE_HISTORY
GROUP BY 1
ORDER BY total_credits DESC
LIMIT 20

## 4. Hourly Usage Pattern
Reveals which hours of the day see peak activity — all-time aggregate.

In [ ]:
%%sql -r hourly_pattern
SELECT
    HOUR(USAGE_TIME)                        AS hour_of_day,
    COUNT(*)                                AS total_requests,
    SUM(TOKENS)                             AS total_tokens,
    ROUND(SUM(TOKEN_CREDITS), 4)            AS total_credits
FROM SNOWFLAKE.ACCOUNT_USAGE.CORTEX_CODE_CLI_USAGE_HISTORY
GROUP BY 1
ORDER BY 1

## 5. Usage by Model
Breaks out credits by model using the `CREDITS_GRANULAR` variant column, with cache read/write split.

In [ ]:
%%sql -r usage_by_model
SELECT
    f.key                                                   AS model,
    COUNT(*)                                                AS total_requests,
    SUM(NVL(f.value:cache_read_input::FLOAT, 0))            AS cache_read_credits,
    SUM(NVL(f.value:cache_write_input::FLOAT, 0))           AS cache_write_credits,
    SUM(NVL(f.value:input::FLOAT, 0))                       AS input_credits,
    SUM(NVL(f.value:output::FLOAT, 0))                      AS output_credits,
    ROUND(
        SUM(NVL(f.value:cache_read_input::FLOAT, 0)
            + NVL(f.value:cache_write_input::FLOAT, 0)
            + NVL(f.value:input::FLOAT, 0)
            + NVL(f.value:output::FLOAT, 0)), 4)            AS total_credits,
    ROUND(
        SUM(NVL(f.value:cache_read_input::FLOAT, 0)
            + NVL(f.value:cache_write_input::FLOAT, 0)
            + NVL(f.value:input::FLOAT, 0)
            + NVL(f.value:output::FLOAT, 0)) * 2.00, 2)     AS estimated_cost_usd
FROM SNOWFLAKE.ACCOUNT_USAGE.CORTEX_CODE_CLI_USAGE_HISTORY,
    LATERAL FLATTEN(input => CREDITS_GRANULAR) f
GROUP BY 1
ORDER BY total_credits DESC

## 6. Cost Projections
Takes the 22 busiest working days on record (proxy for a full working month), then projects to weekly, monthly, and annual spend.

In [ ]:
%%sql -r cost_projections
WITH daily_costs AS (
    SELECT
        USAGE_TIME::DATE            AS usage_date,
        SUM(TOKEN_CREDITS)          AS daily_credits
    FROM SNOWFLAKE.ACCOUNT_USAGE.CORTEX_CODE_CLI_USAGE_HISTORY
    GROUP BY 1
    ORDER BY daily_credits DESC
    LIMIT 22
),
stats AS (
    SELECT
        MIN(daily_credits)  AS min_daily,
        AVG(daily_credits)  AS mean_daily,
        MAX(daily_credits)  AS max_daily
    FROM daily_costs
)
SELECT 'Per Day'   AS period,
    ROUND(min_daily, 2)             AS min_credits,
    ROUND(mean_daily, 2)            AS mean_credits,
    ROUND(max_daily, 2)             AS max_credits,
    ROUND(min_daily * 2.00, 2)      AS min_cost_usd,
    ROUND(mean_daily * 2.00, 2)     AS mean_cost_usd,
    ROUND(max_daily * 2.00, 2)      AS max_cost_usd
FROM stats
UNION ALL
SELECT 'Per Week',
    ROUND(min_daily * 5, 2),  ROUND(mean_daily * 5, 2),  ROUND(max_daily * 5, 2),
    ROUND(min_daily * 5 * 2.00, 2), ROUND(mean_daily * 5 * 2.00, 2), ROUND(max_daily * 5 * 2.00, 2)
FROM stats
UNION ALL
SELECT 'Per Month',
    ROUND(min_daily * 22, 2), ROUND(mean_daily * 22, 2), ROUND(max_daily * 22, 2),
    ROUND(min_daily * 22 * 2.00, 2), ROUND(mean_daily * 22 * 2.00, 2), ROUND(max_daily * 22 * 2.00, 2)
FROM stats
UNION ALL
SELECT 'Per Year',
    ROUND(min_daily * 260, 2), ROUND(mean_daily * 260, 2), ROUND(max_daily * 260, 2),
    ROUND(min_daily * 260 * 2.00, 2), ROUND(mean_daily * 260 * 2.00, 2), ROUND(max_daily * 260 * 2.00, 2)
FROM stats

## 7. Cortex Code Model Pricing Reference
Official rates from the Snowflake Service Consumption Table, **Table 6(e) — Cortex Code** (effective April 1, 2026).  
All rates are AI Credits per one million tokens. Multiply by your AI credit price to get USD cost per 1M tokens.

In [ ]:
import pandas as pd

models = [
    {"model": "claude-4-sonnet",   "input": 1.50, "output": 7.50,  "cache_write": 1.88, "cache_read": 0.15},
    {"model": "claude-opus-4-5",   "input": 2.75, "output": 13.75, "cache_write": 3.44, "cache_read": 0.28},
    {"model": "claude-opus-4-6",   "input": 2.75, "output": 13.75, "cache_write": 3.44, "cache_read": 0.28},
    {"model": "claude-sonnet-4-5", "input": 1.65, "output": 8.25,  "cache_write": 2.07, "cache_read": 0.17},
    {"model": "claude-sonnet-4-6", "input": 1.65, "output": 8.25,  "cache_write": 2.07, "cache_read": 0.17},
    {"model": "openai-gpt-5.2",    "input": 0.97, "output": 7.70,  "cache_write": None, "cache_read": 0.10},
    {"model": "openai-gpt-5.4",    "input": 1.38, "output": 8.25,  "cache_write": None, "cache_read": 0.14},
]

df = pd.DataFrame(models)
df.columns = [
    "Model",
    "Input (AI Credits/1M tokens)",
    "Output (AI Credits/1M tokens)",
    "Cache Write (AI Credits/1M tokens)",
    "Cache Read (AI Credits/1M tokens)",
]

print(f"Source: Snowflake Service Consumption Table, Table 6(e) — Cortex Code (effective April 1, 2026)")
print(f"AI Credit On-Demand Price: ${ai_credit_price:.2f}/credit (global)")
print()
df